## Pipeline ETL: Exemplos de Análise com MovieLens

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.default.ml_25m_data;

### Download de dados brutos

In [0]:
import urllib.request
import ssl
import zipfile

url = "https://files.grouplens.org/datasets/movielens/ml-25m.zip"
vol_path = "/Volumes/workspace/default/ml_25m_data"
zip_path = f"{vol_path}/ml-25m.zip"

# GroupLens certificate workaround
ssl_context = ssl._create_unverified_context()
opener = urllib.request.build_opener(
    urllib.request.HTTPSHandler(context=ssl_context)
)
urllib.request.install_opener(opener)

# Download directly to the Volume
urllib.request.urlretrieve(url, zip_path)

# Extract directly in the Volume
with zipfile.ZipFile(zip_path) as z:
    z.extractall(vol_path)

data_path = f"{vol_path}/ml-25m"

### Extract: leitura dos dados brutos

In [0]:
ratings = spark.read.csv(
    f"{data_path}/ratings.csv",
    header=True,
    inferSchema=True
)

movies = spark.read.csv(
    f"{data_path}/movies.csv",
    header=True,
    inferSchema=True
)

genome_scores = spark.read.csv(
    f"{data_path}/genome-scores.csv",
    header=True,
    inferSchema=True
)

genome_tags = spark.read.csv(
    f"{data_path}/genome-tags.csv",
    header=True,
    inferSchema=True
)

In [0]:
ratings.write.mode("overwrite").saveAsTable("workspace.default.ratings")
movies.write.mode("overwrite").saveAsTable("workspace.default.movies")
genome_scores.write.mode("overwrite").saveAsTable("workspace.default.genome_scores")
genome_tags.write.mode("overwrite").saveAsTable("workspace.default.genome_tags")

### Inspeção básica de tabelas

In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA default;

SHOW TABLES;

In [0]:
%sql
SELECT * FROM movies LIMIT 10;

In [0]:
%sql
SELECT * FROM ratings LIMIT 10;

## Consultas básicas

### Generos associados a cada filme

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW movie_genres AS
SELECT
    movieId,
    title,
    explode(split(genres, '\\|')) AS genre
FROM movies;

In [0]:
%sql
SELECT * FROM movie_genres LIMIT 20;

### Análise estatística: contagem e média de avaliações

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW movie_stats AS
SELECT
    g.genre,
    g.movieId,
    g.title,
    COUNT(*) AS num_ratings,
    ROUND(AVG(r.rating),2) AS avg_rating
FROM movie_genres g
JOIN ratings r
    ON g.movieId = r.movieId
GROUP BY g.genre, g.movieId, g.title
HAVING COUNT(*) >= 1000;

### Filmes melhor avaliados por genero

In [0]:
%sql
WITH ranked_movies AS (
    SELECT
        *,
        RANK() OVER (
            PARTITION BY genre
            ORDER BY avg_rating DESC
        ) AS rank
    FROM movie_stats
)

SELECT *
FROM ranked_movies
WHERE rank <= 5
ORDER BY genre, rank, title;

### Tag Genome

In [0]:
%sql
SELECT * FROM genome_tags LIMIT 100;

In [0]:
%sql
SELECT * FROM genome_scores LIMIT 10;

### Tags associados a um filme específico

In [0]:
%sql
SELECT
    m.title,
    t.tag,
    g.relevance
FROM movies m
JOIN genome_scores g
    ON m.movieId = g.movieId
JOIN genome_tags t
    ON g.tagId = t.tagId
WHERE m.title = 'Blade Runner (1982)'
ORDER BY g.relevance DESC
LIMIT 20;

### Tabelas filtradas, apenas com filmes com mais avaliações

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW popular_movies AS
SELECT
    m.movieId,
    m.title,
    m.genres,
    COUNT(*) AS num_ratings
FROM movies m
JOIN ratings r
    ON m.movieId = r.movieId
GROUP BY
    m.movieId,
    m.title,
    m.genres
ORDER BY num_ratings DESC
LIMIT 1000;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW popular_genome_scores AS
SELECT g.*
FROM genome_scores g
JOIN popular_movies p
    ON g.movieId = p.movieId;

## Similaridade entre Filmes e Recomendações

O **Tag Genome** associa cada filme a várias características (tags).

Para cada par filme-tag, existe um valor de **relevância**. Assim, podemos pensar em cada filme como um vetor:

    Filme A = [relevância_tag1, relevância_tag2, ..., relevância_tagN]
    Filme B = [relevância_tag1, relevância_tag2, ..., relevância_tagN]

Filmes com vetores semelhantes possuem características semelhantes.

### Similaridade de Cosseno

Uma forma comum de comparar dois vetores é a similaridade de cosseno.

Em pseudocódigo:

    dot_product = SUM(A[i] * B[i])

    norm_A = SQRT(SUM(A[i] * A[i]))
    norm_B = SQRT(SUM(B[i] * B[i]))

    similarity = dot_product / (norm_A * norm_B)

Como os valores de relevância do Tag Genome são positivos:

- similaridade próxima de 1 indica filmes muito semelhantes;
- valores menores indicam menor similaridade.

Para calcular isso com SQL, precisamos:

1. alinhar as mesmas tags dos dois filmes;
2. multiplicar os valores de relevância correspondentes;
3. somar esses produtos;
4. calcular a norma de cada vetor;
5. dividir o produto escalar pelo produto das normas.

Vamos fazer isso usando apenas `JOIN`, agregações e operações matemáticas em SQL.

### Cálculo da norma de características para cada filme

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW movie_norms AS
SELECT
    movieId,
    SQRT(SUM(relevance * relevance)) AS norm
FROM popular_genome_scores
GROUP BY movieId;

In [0]:
%sql
SELECT *
from movie_norms
LIMIT 5

### Comparação de características entre dois filmes

In [0]:
%sql
WITH alien AS (
    SELECT g.tagId, g.relevance
    FROM genome_scores g
    JOIN movies m ON g.movieId = m.movieId
    WHERE m.title = 'Alien (1979)'
),
aliens AS (
    SELECT g.tagId, g.relevance
    FROM genome_scores g
    JOIN movies m ON g.movieId = m.movieId
    WHERE m.title = 'Aliens (1986)'
)

SELECT
    a.tagId,
    a.relevance AS alien,
    b.relevance AS aliens,
    a.relevance * b.relevance AS product
FROM alien a
JOIN aliens b ON a.tagId = b.tagId
ORDER BY a.tagId
LIMIT 20;

### Exemplo de Produto escalar entre dois filmes

In [0]:
%sql
WITH alien AS (
    SELECT g.tagId, g.relevance
    FROM genome_scores g
    JOIN movies m ON g.movieId = m.movieId
    WHERE m.title = 'Alien (1979)'
),
aliens AS (
    SELECT g.tagId, g.relevance
    FROM genome_scores g
    JOIN movies m ON g.movieId = m.movieId
    WHERE m.title = 'Aliens (1986)'
)

SELECT
    SUM(a.relevance * b.relevance) AS dot_product
FROM alien a
JOIN aliens b ON a.tagId = b.tagId;

### Produto escalar entre todos pares de filmes

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW movie_dot_products AS
SELECT
    a.movieId AS movieA,
    b.movieId AS movieB,
    SUM(a.relevance * b.relevance) AS dot_product
FROM popular_genome_scores a
JOIN popular_genome_scores b
    ON a.tagId = b.tagId
   AND a.movieId < b.movieId
GROUP BY a.movieId, b.movieId;

In [0]:
%sql 
SELECT *
from movie_dot_products
LIMIT 10

### Cálculo de Similaridade

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW movie_similarities AS
SELECT
    d.movieA,
    d.movieB,
    d.dot_product / (na.norm * nb.norm) AS similarity
FROM movie_dot_products d
JOIN movie_norms na
    ON d.movieA = na.movieId
JOIN movie_norms nb
    ON d.movieB = nb.movieId;

In [0]:
%sql
SELECT
    ma.title AS movieA,
    mb.title AS movieB,
    ROUND(s.similarity, 3) AS similarity
FROM movie_similarities s
JOIN movies ma
    ON s.movieA = ma.movieId
JOIN movies mb
    ON s.movieB = mb.movieId
ORDER BY similarity DESC
LIMIT 20;

### Recomendação de Filmes similares


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW movie_recommendations AS

SELECT
    movieA AS movieId,
    movieB AS similarMovieId,
    similarity
FROM movie_similarities

UNION ALL

SELECT
    movieB AS movieId,
    movieA AS similarMovieId,
    similarity
FROM movie_similarities;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW ranked_recommendations AS
SELECT
    *,
    RANK() OVER (
        PARTITION BY movieId
        ORDER BY similarity DESC
    ) AS rank
FROM movie_recommendations;

In [0]:
%sql
SELECT
    m2.title,
    ROUND(r.similarity, 3) AS similarity
FROM ranked_recommendations r
JOIN movies m1
    ON r.movieId = m1.movieId
JOIN movies m2
    ON r.similarMovieId = m2.movieId
WHERE m1.title = 'Forrest Gump (1994)'
  AND r.rank <= 10
ORDER BY r.rank;

### Similaridade por gêneros implementada por user-defined funcion (UDF)



Podemos definir uma medida mais simples de similaridade usando apenas os gêneros:

    similarity = gêneros_em_comum / gêneros_distintos

A lógica de conjuntos é facilmente expressa em Python e pode ser registrada como
uma função definida pelo usuário (UDF) para uso em consultas SQL.

In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType

def genre_similarity(genres_a, genres_b):
    a = set(genres_a.split("|"))
    b = set(genres_b.split("|"))

    return len(a & b) / len(a | b)

spark.udf.register(
    "genre_similarity",
    genre_similarity,
    DoubleType()
)

In [0]:
%sql
SELECT
    a.title AS movieA,
    b.title AS movieB,
    genre_similarity(a.genres, b.genres) AS genre_similarity
FROM popular_movies a
JOIN popular_movies b
    ON a.movieId <> b.movieId
WHERE a.title = 'Iron Man (2008)'
ORDER BY genre_similarity DESC
LIMIT 10;